# Велодорожки и ДТП в Москве

Предобработка для `analysis.R`: велосипедные ДТП, дороги OSM, сетка радиусом 500 м и расстояния до велодорожек. Запуск из корня проекта: `python scripts/run_preprocessing.py`.

Сравниваем полные 2015 и 2025 годы. Наличие велодорожек определяется по текущему слою: даты открытия большинства линий неизвестны. Подробнее о данных и ограничениях — в `docs/data.md` и `docs/methodology.md`.

In [ ]:
from pathlib import Path
import json
import math
import hashlib
import warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import Polygon, shape
import osmnx as ox

ROOT = Path.cwd()
if not (ROOT / "moskva.geojson").exists():
    raise FileNotFoundError("Откройте notebook из корня проекта с moskva.geojson")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
CACHE = ROOT / "osm_cache"
CACHE.mkdir(exist_ok=True)
CRS = "EPSG:32637"  # UTM 37N, метры; Москва
RADIUS_M = 500.0
PRE_YEAR, POST_YEAR = 2015, 2025
MONTHS = list(range(1, 13))  # одинаковые месяцы в обоих годах
MIN_BIKE_M = 10.0
ROAD_BUFFER_M = 30.0
# Контур: можно заменить локальным GeoJSON нужной территории.
BOUNDARY_FILE = ROOT / "moscow_boundary.geojson"
ROAD_FILE = CACHE / "roads_moscow_admin.geojson"
# Автомобильная УДС: pedestrian/path/cycleway отдельно не включены.
HIGHWAYS = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link",
    "secondary",
    "secondary_link",
    "tertiary",
    "tertiary_link",
    "unclassified",
    "residential",
    "living_street",
    "service",
]
assert PRE_YEAR < POST_YEAR and len(MONTHS) == len(set(MONTHS))
assert MONTHS and all(1 <= m <= 12 for m in MONTHS)
ox.settings.use_cache = True
ox.settings.cache_folder = str(CACHE / "requests")
ox.settings.requests_timeout = 300

## 1. Проверка ДТП и даты
Вложенные сведения об участниках читаются из JSON. Велосипедные ДТП определяются по категории происшествия, категории участников или типу ТС. Это число ДТП с участием велосипедистов, а не число пострадавших велосипедистов. `deaths`/`injuries` — все пострадавшие в происшествии.

Пропущенные и ошибочные даты/геометрии исключаются с отчётом; конфликтующие дубликаты ID останавливают обработку. Нулевые значения появятся только после построения полной панели ячейка × год.

Отдельные `bicycle_crashes.geojson` и `bicycle_crashes.csv` содержат все велосипедные ДТП с корректными точками/датами до фильтрации дорожных ячеек; `bicycle_crashes_assigned.csv` — только назначенные выбранным гексагонам.

`bicycle_crashes_raw.geojson` — полная отфильтрованная исходная выборка, включая ДТП без геометрии, со всеми вложенными полями.


In [ ]:
raw = json.loads((ROOT / "moskva.geojson").read_text())


def is_bicycle_crash(p):
    return (
        "велосипед" in str(p.get("category", "")).lower()
        or any(
            "велосипед" in str(v).lower() for v in p.get("participant_categories") or []
        )
        or any(
            "велосипед" in str(v.get("category", "")).lower()
            for v in p.get("vehicles") or []
        )
    )


# Полная исходная выборка: сохраняем вложенные поля и ДТП без геометрии.
raw_bicycle = {k: v for k, v in raw.items() if k != "features"}
raw_bicycle["features"] = [
    f for f in raw["features"] if is_bicycle_crash(f.get("properties") or {})
]
(OUT / "bicycle_crashes_raw.geojson").write_text(
    json.dumps(raw_bicycle, ensure_ascii=False)
)
records, geometries = [], []
seen = {}
qa = {
    "raw_accidents": len(raw["features"]),
    "duplicate_ids_removed": 0,
    "invalid_point_geometry": 0,
    "bicycle_crashes_raw": len(raw_bicycle["features"]),
}
for feature in raw["features"]:
    p = feature.get("properties") or {}
    uid = p.get("id", p.get("gibdd_number"))
    signature = json.dumps(feature, ensure_ascii=False, sort_keys=True)
    if uid is not None and str(uid) in seen:
        if seen[str(uid)] != signature:
            raise ValueError(f"Конфликтующие записи ДТП ID={uid}")
        qa["duplicate_ids_removed"] += 1
        continue
    if uid is not None:
        seen[str(uid)] = signature
    geom = feature.get("geometry")
    try:
        geom = shape(geom) if geom else None
    except (ValueError, TypeError):
        geom = None
    if geom is None or geom.is_empty or geom.geom_type != "Point" or not geom.is_valid:
        qa["invalid_point_geometry"] += 1
        continue
    bicycle = is_bicycle_crash(p)
    records.append(
        {
            "crash_id": str(uid) if uid is not None else f"row_{len(records)}",
            "datetime": p.get("datetime"),
            "bike_crashes": int(bicycle),
            "deaths": p.get("dead_count"),
            "injuries": p.get("injured_count"),
        }
    )
    geometries.append(geom)
# GeoJSON RFC 7946: WGS84. Если источник объявляет другую CRS, используем её.
source_crs = (raw.get("crs") or {}).get("properties", {}).get("name", "EPSG:4326")
acc = gpd.GeoDataFrame(records, geometry=geometries, crs=source_crs)
acc["datetime"] = pd.to_datetime(
    acc["datetime"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
qa["invalid_dates"] = int(acc.datetime.isna().sum())
acc = acc.loc[acc.datetime.notna()].copy()
for c in ["deaths", "injuries"]:
    acc[c] = pd.to_numeric(acc[c], errors="coerce")
    if acc[c].isna().any() or (acc[c] < 0).any():
        raise ValueError(
            f"Пропущенное/некорректное число {c}; задайте явную политику обработки"
        )
acc["year"] = acc.datetime.dt.year
acc["month"] = acc.datetime.dt.month
coverage = acc.groupby(["year", "month"]).size().rename("crashes").reset_index()
coverage.to_csv(OUT / "coverage_by_month.csv", index=False)
missing_months = {
    int(y): sorted(set(range(1, 13)) - set(g.month))
    for y, g in coverage.groupby("year")
}
qa["months_without_records"] = missing_months
print("Месяцы без записей:", missing_months)
print(
    acc.groupby("year").agg(
        crashes=("crash_id", "size"),
        first=("datetime", "min"),
        last=("datetime", "max"),
    )
)
for y in [PRE_YEAR, POST_YEAR]:
    missing = set(MONTHS) - set(acc.loc[acc.year.eq(y), "month"])
    if missing:
        raise ValueError(
            f"{y}: нет ДТП за месяцы {sorted(missing)}. Выберите одинаковые MONTHS для обоих лет."
        )
# Отдельный слой до ограничения дорожными гексагонами.
bicycle_crashes = acc.loc[acc.bike_crashes.eq(1)].to_crs(4326).copy()
bicycle_crashes["datetime"] = bicycle_crashes.datetime.dt.strftime("%Y-%m-%d %H:%M:%S")
bicycle_crashes["lon"] = bicycle_crashes.geometry.x
bicycle_crashes["lat"] = bicycle_crashes.geometry.y
bicycle_crashes.to_file(OUT / "bicycle_crashes.geojson", driver="GeoJSON")
bicycle_crashes.drop(columns="geometry").to_csv(
    OUT / "bicycle_crashes.csv", index=False
)
qa["bicycle_crashes_valid_points"] = len(bicycle_crashes)
print("Отдельный слой велосипедных ДТП:", len(bicycle_crashes))
acc = acc.to_crs(CRS)
del raw, seen, records, geometries

## 2. Граница Москвы и дорожная сеть OSM
Скачиваем физические OSM ways по автомобильным классам дорог, а не направленные рёбра графа: противоположные направления не удваивают длину. Внутри каждой ячейки объединяем совпадающие линии. Длина — длина геометрии, не полосо-километры.

Текущая сеть задаёт фиксированную область анализа для обоих лет. Она может включать дороги, которых ещё не было в 2015; это ограничение выборки, а не историческая экспозиция. При наличии исторического слоя замените `ROAD_FILE`. Источник: [OSMnx API](https://osmnx.readthedocs.io/en/stable/user-reference.html), © OpenStreetMap contributors, ODbL.


In [ ]:
if BOUNDARY_FILE.exists():
    boundary = gpd.read_file(BOUNDARY_FILE)
else:
    boundary = ox.geocode_to_gdf(
        "R102269", by_osmid=True
    )  # субъект Москва, включая Новую Москву
    boundary.to_file(BOUNDARY_FILE, driver="GeoJSON")
if boundary.crs is None:
    raise ValueError("У границы не задана CRS")
boundary = boundary.to_crs(CRS)
city = shapely.make_valid(boundary.geometry.union_all())
if city.geom_type not in ["Polygon", "MultiPolygon"]:
    raise ValueError("Нужна полигональная граница Москвы")
print("Площадь территории, км²:", round(city.area / 1e6, 1))
if not 500 < city.area / 1e6 < 4000:
    raise ValueError("Проверьте, что геокодер вернул нужную Москву")

if ROAD_FILE.exists():
    provenance_path = ROAD_FILE.with_suffix(".metadata.json")
    if provenance_path.exists():
        provenance = json.loads(provenance_path.read_text())
        if (
            provenance["boundary_sha256"]
            != hashlib.sha256(BOUNDARY_FILE.read_bytes()).hexdigest()
        ):
            raise ValueError(
                "Кэш дорог относится к другому контуру; задайте новый ROAD_FILE"
            )
        if provenance["highway_classes"] != HIGHWAYS:
            raise ValueError(
                "Кэш дорог использует другие классы HIGHWAYS; задайте новый ROAD_FILE"
            )
    else:
        warnings.warn(
            "Дороги без метаданных: проверьте территорию, классы и дату самостоятельно"
        )
    roads = gpd.read_file(ROAD_FILE)
else:
    city_wgs = gpd.GeoSeries([city], crs=CRS).to_crs(4326).iloc[0]
    downloaded = ox.features_from_polygon(city_wgs, tags={"highway": HIGHWAYS})
    downloaded = downloaded.loc[
        downloaded.geometry.geom_type.isin(["LineString", "MultiLineString"])
    ]
    roads = downloaded[["geometry"]].reset_index(drop=True)
    roads.to_file(ROAD_FILE, driver="GeoJSON")
    ROAD_FILE.with_suffix(".metadata.json").write_text(
        json.dumps(
            {
                "downloaded_utc": datetime.now(timezone.utc).isoformat(),
                "source": "OpenStreetMap / Overpass via OSMnx",
                "highway_classes": HIGHWAYS,
                "boundary_sha256": hashlib.sha256(
                    BOUNDARY_FILE.read_bytes()
                ).hexdigest(),
            },
            indent=2,
        )
    )
if roads.crs is None:
    raise ValueError("У дорог не задана CRS")
roads = roads.to_crs(CRS)
roads = roads.loc[roads.geometry.notna() & ~roads.geometry.is_empty].copy()
roads["geometry"] = shapely.make_valid(roads.geometry)
# Обрезаем линии при пересечении с каждой ячейкой ниже.
# Здесь достаточно отбора: дорогое пересечение каждой дороги со всем
# сложным контуром Москвы не требуется.
roads = roads.iloc[roads.sindex.query(city, predicate="intersects")].copy()
roads = roads.explode(index_parts=False, ignore_index=True)
roads = roads.loc[
    roads.geometry.geom_type.eq("LineString") & roads.length.gt(0)
].reset_index(drop=True)
if roads.empty:
    raise ValueError("Дорожная сеть пуста")
print("Линейных объектов дорог:", len(roads))

## 3. Сетка и длины инфраструктуры

Радиус 500 м измеряется от центра до вершины. Сетка закреплена в UTM; идентификаторы ячеек не зависят от порядка строк. На границе Москвы ячейки обрезаются. В выборку входят ячейки, пересекающие автомобильные дороги, в том числе касающиеся их в одной точке.

`bike_length_m` — длина текущей велоинфраструктуры в ячейке. Группа с велодорожками задана порогом 10 м. `bike_near_road_m` дополнительно ограничивает линии коридором 30 м от дорог; показатель сохранён для дальнейших проверок.

In [ ]:
def hex_grid(polygon, radius=500.0):
    xmin, ymin, xmax, ymax = polygon.bounds
    dx, dy = 1.5 * radius, math.sqrt(3) * radius
    rows = []
    for col in range(math.floor(xmin / dx) - 1, math.ceil(xmax / dx) + 2):
        shift = (col % 2) * dy / 2
        for row in range(
            math.floor((ymin - shift) / dy) - 1, math.ceil((ymax - shift) / dy) + 2
        ):
            x, y = col * dx, row * dy + shift
            angles = np.arange(6) * math.pi / 3
            full = Polygon(
                np.column_stack(
                    [x + radius * np.cos(angles), y + radius * np.sin(angles)]
                )
            )
            clipped = full.intersection(polygon)
            if not clipped.is_empty and clipped.area > 1:
                rows.append(
                    {
                        "hex_id": f"h_{col}_{row}",
                        "center_x": x,
                        "center_y": y,
                        "area_m2": clipped.area,
                        "geometry": clipped,
                    }
                )
    return gpd.GeoDataFrame(rows, crs=CRS)


all_hex = hex_grid(city, RADIUS_M)


def local_union(layer, polygon):
    idx = layer.sindex.query(polygon, predicate="intersects")
    if len(idx) == 0:
        return shapely.GeometryCollection()
    return layer.geometry.iloc[idx].union_all().intersection(polygon)


lengths, road_hit = [], []
for geom in all_hex.geometry:
    line = local_union(roads, geom)
    lengths.append(line.length)
    road_hit.append(not line.is_empty)
all_hex["road_length_m"] = lengths
all_hex["intersects_road"] = road_hit
hexes = all_hex.loc[all_hex.intersects_road].copy().reset_index(drop=True)
qa["grid_cells_total"] = len(all_hex)
qa["grid_cells_with_roads"] = len(hexes)

bike = gpd.read_file(ROOT / "bike_lanes_singlepart.geojson")
if bike.crs is None:
    raise ValueError("У велодорожек не задана CRS")
bike = bike.to_crs(CRS)
bike = bike.loc[bike.geometry.notna() & ~bike.geometry.is_empty].copy()
bike["geometry"] = shapely.make_valid(bike.geometry)
bike = gpd.clip(bike, gpd.GeoSeries([city], crs=CRS), keep_geom_type=True)
bike = bike.loc[
    bike.geometry.geom_type.isin(["LineString", "MultiLineString"]) & bike.length.gt(0)
]
if bike.empty:
    raise ValueError("Велодорожки не пересекают выбранную территорию")
bike_union = bike.geometry.union_all()
bike_length, near_road_length = [], []
for geom in hexes.geometry:
    b = local_union(bike, geom)
    # Дополнительные дороги вне ячейки нужны для буфера возле её границы.
    bike_length.append(b.length)
    if b.is_empty or b.length == 0:
        near_road_length.append(0.0)
    else:
        r = local_union(roads, geom.buffer(ROAD_BUFFER_M))
        near_road_length.append(b.intersection(r.buffer(ROAD_BUFFER_M)).length)
hexes["bike_length_m"] = bike_length
hexes["bike_near_road_m"] = near_road_length
hexes["treated_current"] = (hexes.bike_length_m >= MIN_BIKE_M).astype(int)
hexes["distance_to_bike_m"] = hexes.geometry.distance(bike_union)
centers = gpd.GeoSeries(
    gpd.points_from_xy(hexes.center_x, hexes.center_y), crs=CRS
).to_crs(4326)
hexes["lon"], hexes["lat"] = centers.x, centers.y
# Блоки 5 км для локальных сравнений и кластеризации ошибок.
hexes["spatial_block"] = (
    np.floor(hexes.center_x / 5000).astype(int).astype(str)
    + "_"
    + np.floor(hexes.center_y / 5000).astype(int).astype(str)
)
hexes.to_file(OUT / "hexagons_500m.gpkg", layer="hexagons", driver="GPKG")
print(
    hexes.groupby("treated_current").agg(
        hexagons=("hex_id", "size"),
        roads_km=("road_length_m", lambda x: x.sum() / 1000),
    )
)

## 4. Однозначное назначение ДТП и полная панель
Для точки на общей границе выбираем минимальный ID: одно ДТП не попадёт в две ячейки. ДТП в ячейках без УДС исключаются отдельно с числом потерь. Точки не привязываются к ближайшей дороге: это сохранило бы ошибочные координаты и могло бы переместить исход между treatment и control.

Все годы выгружаются для описательного графика, но R использует только два выбранных года и одинаковые месяцы. Промежуточные годы не считаются автоматически pre/post: время treatment неизвестно.


In [ ]:
inside = acc.geometry.intersects(city)
qa["accidents_outside_city"] = int((~inside).sum())
acc_city = acc.loc[inside].copy().reset_index(drop=True)
acc_city["point_row"] = np.arange(len(acc_city))
joined = gpd.sjoin(
    acc_city, all_hex[["hex_id", "geometry"]], how="left", predicate="intersects"
)
qa["boundary_multiple_matches"] = int(joined.point_row.duplicated().sum())
assigned = joined.sort_values(["point_row", "hex_id"]).drop_duplicates("point_row")
qa["inside_city_unassigned"] = int(assigned.hex_id.isna().sum())
if qa["inside_city_unassigned"]:
    raise ValueError("ДТП внутри города не назначены сетке; проверьте геометрию")
qa["accidents_in_cells_without_roads"] = int(
    (~assigned.hex_id.isin(hexes.hex_id)).sum()
)
assigned = assigned.loc[assigned.hex_id.isin(hexes.hex_id)].copy()
assigned[
    [
        "crash_id",
        "hex_id",
        "datetime",
        "year",
        "month",
        "bike_crashes",
        "deaths",
        "injuries",
    ]
].to_csv(OUT / "accidents_assigned.csv", index=False)
assigned.loc[
    assigned.bike_crashes.eq(1),
    ["crash_id", "hex_id", "datetime", "year", "month", "deaths", "injuries"],
].to_csv(OUT / "bicycle_crashes_assigned.csv", index=False)
selected = assigned.loc[assigned.month.isin(MONTHS)].copy()
selected["crashes"] = 1
outcomes = ["crashes", "bike_crashes", "deaths", "injuries"]
counts = selected.groupby(["hex_id", "year"])[outcomes].sum()
years = sorted(set(acc.year))
index = pd.MultiIndex.from_product([hexes.hex_id, years], names=["hex_id", "year"])
panel = counts.reindex(index, fill_value=0).reset_index()
static = pd.DataFrame(hexes.drop(columns="geometry"))
panel = panel.merge(static, on="hex_id", how="left", validate="many_to_one")
panel["post"] = (panel.year == POST_YEAR).astype(int)
panel["did_current"] = panel.treated_current * panel.post
panel["months_observed"] = (
    panel.year.map(
        coverage.loc[coverage.month.isin(MONTHS)].groupby("year").month.nunique()
    )
    .fillna(0)
    .astype(int)
)
panel["period_months"] = len(MONTHS)
panel["is_endpoint"] = panel.year.isin([PRE_YEAR, POST_YEAR]).astype(int)
panel.to_csv(OUT / "hex_panel.csv", index=False)
assert not panel.duplicated(["hex_id", "year"]).any()
assert panel.loc[panel.is_endpoint.eq(1)].shape[0] == 2 * len(hexes)
assert int(panel.crashes.sum()) == len(selected)
for c in outcomes:
    assert int(panel[c].sum()) == int(selected[c].sum())
qa["accidents_assigned"] = len(assigned)
qa["accidents_selected_months"] = len(selected)
print(json.dumps(qa, ensure_ascii=False, indent=2))

## 5. Настройки и историческая разметка

В `metadata.json` сохраняются параметры, версии пакетов, контрольные суммы файлов и потери при обработке.

`treatment_audit.csv` — шаблон для будущей проверки состояния ячеек в обоих годах. `bike_pre` и `bike_post` принимают 0, 1 или пустое значение; в `evidence` указывается источник. Текущий R-расчёт этот шаблон не использует. Частичная проверка отдельных линий хранится отдельно в `data/history_evidence.json`.

Без достоверных состояний на обоих концах периода нельзя выделить эффект строительства. Для staggered DiD дополнительно нужны годы открытия.

In [ ]:
audit_path = OUT / "treatment_audit.csv"
if not audit_path.exists():  # не перезаписываем ручную проверку
    audit = static[
        ["hex_id", "bike_length_m", "bike_near_road_m", "treated_current"]
    ].copy()
    audit["bike_pre"] = pd.NA
    audit["bike_post"] = pd.NA
    audit["evidence"] = ""
    audit.to_csv(audit_path, index=False)
metadata = {
    "pre_year": PRE_YEAR,
    "post_year": POST_YEAR,
    "months": MONTHS,
    "radius_m": RADIUS_M,
    "radius_definition": "center_to_vertex",
    "metric_crs": CRS,
    "min_bike_m": MIN_BIKE_M,
    "road_buffer_m": ROAD_BUFFER_M,
    "bike_snapshot_date": None,
    "road_snapshot": "current_or_user_supplied",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "qa": qa,
    "input_sha256": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in [
            ROOT / "moskva.geojson",
            ROOT / "bike_lanes_singlepart.geojson",
            BOUNDARY_FILE,
            ROAD_FILE,
        ]
    },
    "versions": {
        "pandas": pd.__version__,
        "geopandas": gpd.__version__,
        "shapely": shapely.__version__,
        "osmnx": ox.__version__,
    },
}
(OUT / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
print("Готово. Запустите из корня проекта: Rscript analysis.R")

## 6. Велодорожки и соседние территории

Сохраняем расстояния от границы и центра каждого гексагона до текущих велодорожек. В R используются соседние кольца 0–500, 500–1000 и 1000–2000 м; удалённый контроль находится дальше 2 км. Это прямые расстояния, а не расстояния по дорожной сети.

Отдельно считаем точечные велосипедные ДТП в коридорах ≤30, 30–500, 500–1000, 1000–2000 и >2000 м. Каждая точка относится только к одной зоне. Так можно увидеть изменения внутри treated-гексагона; близость к велодорожке не доказывает, что ДТП произошло на ней.

`spillover_analysis.R` сравнит соседние группы и суммарный исход для велодорожки вместе с окружением. Все оценки остаются описательными до проверки исторического treatment.

In [ ]:
from scripts.spatial_spillover import prepare_spillovers

spillover_exposure = prepare_spillovers(ROOT)